# Khối C: Huấn luyện DenseNet-121 (đa nhãn 14 bệnh)

Đầu vào: `train.csv`, `val.csv`, `test.csv` (Khối B) và `images_224.npy`. Chạy lại toàn bộ notebook sau khi Colab ngắt kết nối thì sẽ tự tiếp tục từ epoch dở dang.

## 1. Cấu hình và nối Drive

Ô này chuẩn bị môi trường cho toàn bộ phần huấn luyện: nạp thư viện, nối Google Drive, khai báo đường dẫn và siêu tham số, rồi chọn thiết bị chạy (GPU hoặc CPU).


In [1]:
import os, shutil, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')

# ---- Đường dẫn (sửa nếu khác) ----
DATA_DIR = '/content/drive/MyDrive/CXR_Project/data/'
IMAGES_NPY = DATA_DIR + 'images_224.npy'
RUN_DIR = '/content/drive/MyDrive/CXR_Project/runs/densenet121_baseline/'
os.makedirs(RUN_DIR, exist_ok=True)

# Đọc mmap trực tiếp từ Drive rất chậm khi truy cập ngẫu nhiên.
# True = copy file về đĩa Colab một lần rồi mmap từ đó (nhanh hơn nhiều).
COPY_TO_LOCAL = True

DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia',
]

# ---- Siêu tham số ----
MAX_EPOCHS = 6
PATIENCE = 3          # dừng sớm nếu không tăng sau chừng này epoch liên tiếp
BATCH_SIZE = 32
LR = 1e-4
NUM_WORKERS = 2
SEED = 42

LAST_CKPT = RUN_DIR + 'last.pt'   # trạng thái đầy đủ, ghi sau mỗi epoch (để resume)
BEST_CKPT = RUN_DIR + 'best.pt'   # trọng số tốt nhất theo macro AUROC validation

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'KHÔNG CÓ GPU')

Mounted at /content/drive
Device: cuda | Tesla T4


## 2. Đọc ảnh bằng memory map

Ô này mở file `images_224.npy` ở chế độ memory map (chỉ đọc phần ảnh cần dùng, không nạp cả mảng ~5,6 GB vào RAM), nhận diện định dạng ảnh, rồi đọc ba file CSV và kiểm tra cột `idx`.

In [2]:
assert os.path.exists(IMAGES_NPY), f'Không thấy {IMAGES_NPY}'

if COPY_TO_LOCAL:
    local = '/content/images_224.npy'
    if not (os.path.exists(local) and os.path.getsize(local) == os.path.getsize(IMAGES_NPY)):
        print('Đang copy images_224.npy về đĩa Colab (chỉ lần đầu)...')
        shutil.copyfile(IMAGES_NPY, local)
    IMAGES_NPY = local

imgs = np.load(IMAGES_NPY, mmap_mode='r')
print('Mảng ảnh:', imgs.shape, imgs.dtype)

# Nhận diện bố cục: (N,H,W) / (N,H,W,C) / (N,C,H,W)
if imgs.ndim == 3:
    LAYOUT = 'NHW'
elif imgs.ndim == 4 and imgs.shape[-1] in (1, 3):
    LAYOUT = 'NHWC'
elif imgs.ndim == 4 and imgs.shape[1] in (1, 3):
    LAYOUT = 'NCHW'
else:
    raise ValueError(f'Không nhận diện được bố cục ảnh: {imgs.shape}')

# Thang giá trị: uint8 -> chia 255; float đã chuẩn hóa 0-1 -> giữ nguyên
sample_max = float(np.asarray(imgs[:64]).max())
SCALE = 255.0 if (imgs.dtype == np.uint8 or sample_max > 1.5) else 1.0
print('Bố cục:', LAYOUT, '| thang chia:', SCALE, '| max mẫu:', sample_max)

train_df = pd.read_csv(DATA_DIR + 'train.csv')
val_df = pd.read_csv(DATA_DIR + 'val.csv')
test_df = pd.read_csv(DATA_DIR + 'test.csv')

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    assert 'idx' in d.columns, f'{name}.csv thiếu cột idx'
    assert d['idx'].min() >= 0 and d['idx'].max() < len(imgs), f'idx của {name} vượt phạm vi mảng ảnh'
    assert d['idx'].is_unique, f'{name}.csv có idx trùng'
    print(f'{name}: {len(d)} ảnh')

Đang copy images_224.npy về đĩa Colab (chỉ lần đầu)...
Mảng ảnh: (112120, 224, 224) uint8
Bố cục: NHW | thang chia: 255.0 | max mẫu: 255.0


AssertionError: train.csv thiếu cột idx

## 3. Dataset và DataLoader

Ô này định nghĩa cách đưa ảnh và nhãn vào mô hình: lớp `CXRDataset` đọc từng ảnh từ mảng `.npy`, `make_loader` gom ảnh thành từng batch, và `prep` chuẩn hóa ảnh trước khi đưa vào DenseNet.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

class CXRDataset(Dataset):
    """Lấy ảnh theo cột idx của dòng CSV; thứ tự mẫu = thứ tự dòng CSV."""
    def __init__(self, df):
        self.idx = df['idx'].to_numpy(dtype=np.int64)
        self.y = df[DISEASES].to_numpy(dtype=np.float32)
        self.arr = None  # mở mmap lười trong từng worker

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        if self.arr is None:
            self.arr = np.load(IMAGES_NPY, mmap_mode='r')
        a = np.array(self.arr[self.idx[i]])      # copy khỏi mmap
        if LAYOUT == 'NHW':
            a = a[None]                          # (1,H,W)
        elif LAYOUT == 'NHWC':
            a = a.transpose(2, 0, 1)             # (C,H,W)
        return torch.from_numpy(np.ascontiguousarray(a)), torch.from_numpy(self.y[i])

def make_loader(df, shuffle, seed=None):
    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)
    return DataLoader(CXRDataset(df), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=False, generator=g, drop_last=False)

def prep(x):
    """uint8/float -> float chuẩn hóa ImageNet, 3 kênh, trên GPU."""
    x = x.to(device, non_blocking=True).float() / SCALE
    if x.shape[1] == 1:
        x = x.expand(-1, 3, -1, -1)
    return (x - MEAN) / STD

val_loader = make_loader(val_df, shuffle=False)
test_loader = make_loader(test_df, shuffle=False)

## 4. Mô hình, loss, optimizer

Ô này dựng mô hình DenseNet-121 và các thành phần dùng để huấn luyện nó.

In [ ]:
def build_model():
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, len(DISEASES))  # không sigmoid
    return m

model = build_model().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

## 5. Hàm dự đoán và macro AUROC

Ô này định nghĩa ba hàm dùng chung cho cả huấn luyện lẫn xuất kết quả.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    """Trả về (logits, labels), mảng N x 14, đúng thứ tự dòng CSV."""
    model.eval()
    out_l, out_y = [], []
    for x, y in tqdm(loader, leave=False):
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(prep(x))
        out_l.append(logits.float().cpu().numpy())
        out_y.append(y.numpy())
    return np.concatenate(out_l), np.concatenate(out_y)

def macro_auroc(logits, labels):
    probs = 1 / (1 + np.exp(-logits))
    aucs = []
    for c in range(labels.shape[1]):
        if labels[:, c].min() == labels[:, c].max():   # lớp chỉ có một giá trị: không tính được
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(labels[:, c], probs[:, c]))
    return float(np.nanmean(aucs)), aucs

def atomic_save(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)   # tránh file hỏng nếu bị ngắt giữa chừng

## 6. Huấn luyện (tự resume)

Checkpoint `last.pt` (model, optimizer, scaler, epoch, AUROC tốt nhất, bộ đếm dừng sớm) được ghi sau **mỗi epoch**. Nếu bị ngắt giữa một epoch, epoch đó sẽ chạy lại từ đầu.

In [ ]:
start_epoch, best_auc, no_improve, history, finished = 0, -1.0, 0, [], False

if os.path.exists(LAST_CKPT):
    ck = torch.load(LAST_CKPT, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    best_auc, no_improve = ck['best_auc'], ck['no_improve']
    history, finished = ck['history'], ck['finished']
    print(f'Tiếp tục: đã xong {start_epoch} epoch | best AUROC = {best_auc:.4f} | finished = {finished}')
else:
    print('Chưa có checkpoint: huấn luyện từ đầu')

if finished or start_epoch >= MAX_EPOCHS:
    print('Huấn luyện đã hoàn tất, bỏ qua sang bước dự đoán.')
else:
    for epoch in range(start_epoch, MAX_EPOCHS):
        t0 = time.time()
        # seed theo epoch để thứ tự xáo trộn tái lập được khi resume
        train_loader = make_loader(train_df, shuffle=True, seed=SEED + epoch)
        model.train()
        run_loss, n = 0.0, 0
        for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{MAX_EPOCHS}', leave=False):
            x, y = prep(x), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * x.size(0)
            n += x.size(0)

        v_logits, v_labels = predict(model, val_loader)
        auc, _ = macro_auroc(v_logits, v_labels)
        history.append({'epoch': epoch + 1, 'train_loss': run_loss / n, 'val_auroc': auc})

        if auc > best_auc:
            best_auc, no_improve = auc, 0
            atomic_save({'model': model.state_dict(), 'epoch': epoch, 'val_auroc': auc}, BEST_CKPT)
            tag = '  <- tốt nhất, đã lưu best.pt'
        else:
            no_improve += 1
            tag = f'  (không tăng {no_improve}/{PATIENCE})'

        finished = (no_improve >= PATIENCE) or (epoch + 1 >= MAX_EPOCHS)
        atomic_save({
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scaler': scaler.state_dict(), 'epoch': epoch, 'best_auc': best_auc,
            'no_improve': no_improve, 'history': history, 'finished': finished,
        }, LAST_CKPT)

        print(f'Epoch {epoch+1}: loss={run_loss/n:.4f} | val macro AUROC={auc:.4f} | '
              f'{time.time()-t0:.0f}s{tag}')
        if finished:
            if no_improve >= PATIENCE:
                print('Dừng sớm.')
            break

print(pd.DataFrame(history).round(4).to_string(index=False))